# Bilingual Academic AI Agent
An Agentic AI workflow for bypassing the chaos of foreign university websites. Built with Python, Playwright Async API, and Google Gemini, this tool acts as a bilingual higher-education consultant.

Key Features:
- Autonomous Web Crawling: Fetches entire site structures and intelligently predicts which subpages (like Admissions and Scholarships) hold the highest value.
- Robust Parsing: Built to bypass bots, lazy-loaded JavaScript, and Chinese network firewalls to extract clean text.
- Personalized Synthesis: Cross-references raw, translated university data against a user's academic profile to output highly tailored insights on program eligibility and tuition costs.

## Importing the Necessary Libraries

In [9]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from google import genai

## Loading the Gemini API Key

In [10]:
load_dotenv(override=True)

api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


## Defining the System Prompt

The System Prompt is the core "brain" of our AI Agent. Here, we move beyond generic instructions and give the LLM a highly specialized persona: an **Expert Bilingual Higher Education Consultant**. 

By being explicit about the output structure (using a strict markdown template) and instructing the model to *never* guess or hallucinate missing information, we ensure our agent acts as a reliable data extractor rather than just a creative chatbot. It sifts through the noisy HTML data, translates the relevant Chinese portions, and maps everything into our 6-point checklist.

In [11]:
system_prompt = """
    You are an expert Bilingual Higher Education Consultant and Data Extraction Specialist. Your primary task is to analyze raw, scraped website data (HTML text or markdown) from Chinese universities and transform it into a clear, structured, and comprehensive English brief for a prospective international student.

    ### CORE RULES & CONSTRAINTS
    1. **Zero Hallucination:** You must NEVER invent, guess, or infer information. All extracted data must be strictly grounded in the provided scraped text.
    2. **No External Knowledge:** Even if you independently know a fact about the university (e.g., knowing it is a "985" university or knowing its usual tuition costs), you must not include it unless it is explicitly stated in the provided text.
    3. **Handle Missing Data Explicitly:** If a specific piece of requested information is NOT present in the scraped text, do not guess or attempt to fill in the blanks. You must explicitly write: "Not mentioned in the provided text."
    4. **Faithful Extraction:** Do not convert currencies (e.g., estimating RMB to USD) or alter application dates unless the source text explicitly provides those conversions.

    ### TASK INSTRUCTIONS
    The user will provide you with the scraped text from a university's website. Your goal is to sift through the noise (menus, footers, boilerplate code) and extract actionable, accurate information according to the categories below. 

    Respond using the following exact markdown structure:

    # [University English Name] ([University Chinese Name] / [Pinyin])

    ## 1. Quick Overview
    * **Location:** (City, Province, and specific campuses if mentioned)
    * **University Status:** (Note if they mention being a "985", "211", or "Double First-Class" university)
    * **General Description:** (A 2-3 sentence summary of the university's focus)

    ## 2. Detailed Programs Offered
    * **Degree Levels:** (Specify if the text mentions Bachelor's, Master's, Doctoral/PhD, or Non-degree language programs)
    * **Specific Majors/Programs:** (List the specific programs mentioned in the text. Group them by degree level if possible. Highlight any programs explicitly labeled as English-taught)
    * **Program Duration:** (e.g., 4 years for Bachelor's, 2-3 years for Master's)
    * **Language of Instruction:** (Clearly distinguish which programs are taught in Chinese vs. English)

    ## 3. Admissions & Requirements
    * **Application Deadlines:** (Extract exact dates or general timelines like "Spring/Autumn Intake")
    * **Language Requirements:** (HSK scores for Chinese programs; IELTS/TOEFL scores for English programs)
    * **Required Documents:** (e.g., transcripts, passport copies, recommendation letters, study plan)
    * **Application Portal/Method:** (Specific links or email addresses for submitting applications)

    ## 4. Tuition, Fees, & Scholarships
    * **Application Fee:** (Extract the non-refundable application/registration fee, usually around 400-800 RMB)
    * **Tuition Fee Breakdown:** (Provide exact numerical values. Critically: note if the tuition varies by major—e.g., arts vs. STEM vs. medicine—or by degree level. Specify if the cost is per semester or per year, and ensure the currency is noted, usually RMB)
    * **Accommodation Fees:** (List on-campus dorm costs, noting single vs. double room prices if available)
    * **Available Scholarships:** (Detail every scholarship mentioned. Categorize them if possible:
        * *Chinese Government Scholarship (CSC):* (Note type/coverage if mentioned)
        * *Provincial/City Scholarships:* (e.g., "Shanghai Government Scholarship")
        * *University-Specific Scholarships:* (e.g., "President's Scholarship")
        * *Scholarship Coverage:* (Extract what is actually paid for—does it cover full tuition, accommodation, and a monthly living stipend? Or is it just a partial tuition waiver?)

    ## 5. Contact Information & Next Steps
    * **International Students Office Contact:** (Emails, phone numbers, addresses)
    * **Important Links:** (Application portal, scholarship page, program lists)

    ## 6. Extraction Notes
    * Provide a brief note on any messy or unclear data in the scrape.
    * List 2-3 specific questions or missing pieces of critical information (like missing tuition costs or dead links) the user should try to find on another page.    
"""

## Crafting the User Prompt

The User Prompt is where we dynamically inject our actual data. Because we want this agent to be personalized, we pass in two critical pieces of information:
1. **The Target Data:** The raw, messy text we just scraped from the university website.
2. **The Student Profile:** The user's specific context (like nationality, desired degree, and budget).

In [15]:
def create_user_prompt(website, user_profile=None):
    user_section = ""
    if user_profile:
        user_section = f"""
            ---
            ## MY STUDENT PROFILE
            Use this profile to personalize your analysis. Highlight programs, scholarships,
            and requirements that are most relevant to me.

            - **Nationality:** {user_profile.get('nationality', 'Not specified')}
            - **Desired Degree Level:** {user_profile.get('degree_level', 'Not specified')}
            - **Field of Interest:** {user_profile.get('field', 'Not specified')}
            - **Current Education:** {user_profile.get('current_education', 'Not specified')}
            - **Chinese Proficiency (HSK Level):** {user_profile.get('hsk_level', 'Not specified')}
            - **English Proficiency (IELTS/TOEFL):** {user_profile.get('english_level', 'Not specified')}
            - **Budget Constraint:** {user_profile.get('budget', 'Not specified')}
            - **Scholarship Interest:** {user_profile.get('scholarship_interest', 'Yes')}
        """

    return f"""
        {user_section}
        ---
        ## RAW SCRAPED WEBSITE DATA
        Below is the raw text extracted from a Chinese university's website.
        It may contain navigation menus, repeated headers, footers, or garbled formatting.
        Your job is to extract only the meaningful content.

        {website}
        ---

        ## PROCESSING INSTRUCTIONS
        1. Ignore all boilerplate content (navigation bars, cookie notices, footers, social media links).
        2. Extract and structure the data according to the categories defined in your system instructions.
        3. If any Chinese text (中文) is present, translate it to English and include the original Chinese in parentheses.
        4. Preserve all numerical data exactly as found (tuition costs, dates, phone numbers).
        5. If my student profile is provided, add a "## 7. Personalized Recommendations" section at the end with:
        - Which specific programs match my field of interest
        - Whether I meet the language requirements or what I still need
        - Scholarships I am most likely eligible for
        - Estimated total annual cost after potential scholarship
    """

## Autonomous Agent Workflow

Instead of manually feeding specific links to the scraper, we are giving our agent **autonomy** to navigate the university's site on its own. 

This workflow uses a standard **"Agentic Crawl"** pattern:
1. **Discover:** The agent visits the university homepage and extracts every single link available.
2. **Think & Filter:** It passes the massive list of links to the Gemini 2.0 Flash model, acting as a "Web Navigation Agent," to critically select the 3-5 subpages most likely to contain admissions, program, and scholarship information.
3. **Act:** It autonomously scrapes those selected high-value pages.
4. **Synthesize:** It compiles all the gathered data, combines it with your student profile, and triggers the main prompt to generate your final, personalized admissions brief.


In [13]:
import json
from uni_scraper import fetch_website_links, fetch_multiple_pages
from google.genai import types

async def autonomous_university_researcher(homepage_url, user_profile, client):
    print(f"🕵️‍♂️ Agent starting research on: {homepage_url}")
    
    # Step 1: Get all links from the homepage
    print("🔍 Extracting links from the homepage...")
    all_links = await fetch_website_links(homepage_url)
    
    if not all_links:
        return "Error: Could not retrieve links from the homepage."
        
    # Step 2: Ask the LLM to pick the 3-5 most relevant links
    print(f"🧠 Analyzing {len(all_links)} links to find Admissions, Programs, and Scholarships pages...")
    link_picker_prompt = f"""
    You are a web navigation agent. Below is a list of URLs found on a Chinese university's homepage.
    Select the 3 to 5 URLs that are MOST likely to contain details about:
    1. Degree programs for international students
    2. Admissions requirements and guidelines
    3. Scholarships (CSC, local government, or university scholarships)
    
    Only return a valid JSON list of strings (the URLs). Do not return any other text or markdown blocks.
    
    LINKS:
    {json.dumps(all_links, indent=2)}
    """
    
    link_response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=link_picker_prompt
    )
    
    # Parse the chosen links safely
    try:
        # Clean the response in case the model adds ```json markdown
        cleaned_json = link_response.text.strip().removeprefix('```json').removesuffix('```').strip()
        chosen_links = json.loads(cleaned_json)
        print(f"🎯 Agent selected {len(chosen_links)} high-value pages to scrape:")
        for link in chosen_links:
            print(f"   - {link}")
    except Exception as e:
        print(f"⚠️ Failed to parse LLM link selection. Error: {e}")
        print("Raw response:", link_response.text)
        return None

    # Step 3: Scrape the selected pages
    print("\n📖 Scraping the selected pages (this may take a moment)...")
    combined_text = await fetch_multiple_pages(chosen_links, max_chars=8000)
    
    # Step 4: Generate the final university brief
    print("\n✍️ Compiling the final university brief...")
    final_prompt = create_user_prompt(combined_text, user_profile)
    
    final_response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=final_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt
        )
    )
    
    print("✅ Research Complete!\n")
    return final_response.text


## Main Execution & Testing

In the cell below, we initialize the Gemini API client and define a dummy `my_profile` dictionary. Try changing the profile parameters (like your nationality or desired degree level) and notice how the agent intelligently adapts the final brief—highlighting relevant scholarships or warning you about language requirements based on your input. 

In [ ]:
from google.genai import types

client = genai.Client(api_key=str(api_key))

user_profile = {
    'nationality': 'Indonesian',
    'degree_level': "Master's Degree", 
    'field': 'Computer Science or Data Analytics',
    'current_education': "Bachelor's Degree",
    'hsk_level': 'None (HSK 0)',
    'english_level': 'IELTS 8.0',
    'budget': 'Max 40,000 RMB per year for tuition (if not fully funded)',
    'scholarship_interest': 'Extremely high. Prioritizing Chinese Government Scholarship (CSC) or full university waivers.'
}

brief = await autonomous_university_researcher(
    homepage_url="https://www.hitsz.edu.cn", 
    user_profile=user_profile,
    client=client
)

display(Markdown(brief))

🕵️‍♂️ Agent starting research on: https://www.hitsz.edu.cn
🔍 Extracting links from the homepage...
🧠 Analyzing 147 links to find Admissions, Programs, and Scholarships pages...
🎯 Agent selected 4 high-value pages to scrape:
   - http://en.hitsz.edu.cn
   - http://zsb.hitsz.edu.cn/
   - https://yzb.hitsz.edu.cn/zs
   - https://www.hitsz.edu.cn/xsjz/list.htm

📖 Scraping the selected pages (this may take a moment)...



✍️ Compiling the final university brief...
✅ Research Complete!



# Harbin Institute of Technology, Shenzhen (哈尔滨工业大学（深圳） / Hā'ěrbīn Gōngyè Dàxué Shēnzhèn)

## 1. Quick Overview
*   **Location:** HIT Campus of University Town of Shenzhen, Taoyuan Street, Nanshan District, Shenzhen, Guangdong Province, China.
*   **University Status:** Not explicitly mentioned in the provided text as "985" or "211" (though it is a campus of Harbin Institute of Technology).
*   **General Description:** A research-oriented campus in Shenzhen focusing on high-tech fields such as informatics, computer science, artificial intelligence, and frontier sciences. It maintains strong links with the main Harbin campus and local national laboratories.

## 2. Detailed Programs Offered
*   **Degree Levels:** Master's and Doctoral/PhD (specifically mentioned in the postgraduate admissions site). Undergraduate courses are mentioned via news links.
*   **Specific Majors/Programs:** 
    *   **College of Informatics:** School of Computer Science and Technology, School of Information Science and Technology, School of Integrated Circuits, International Research Institute for Artificial Intelligence, Institute of Cyberspace Security.
    *   **College of Artificial Intelligence:** School of Intelligence Science and Engineering, School of Robotics and Advanced Manufacture, School of Intelligent Civil and Ocean Engineering, Research Institute of Intelligent Ocean Engineering.
    *   **Other Schools:** Materials Science and Engineering, Science, Aerospace Science, Biomedical Engineering, Future Design, Architecture, Economics and Management.
    *   **Specific 2026 Master's Programs mentioned:** Naval Architecture and Ocean Engineering (船舶与海洋工程), Civil and Hydraulic Engineering (土木水利).
*   **Program Duration:** Not mentioned in the provided text.
*   **Language of Instruction:** Not explicitly categorized in the provided text (though several schools focus on internationalized fields like AI and Robotics).

## 3. Admissions & Requirements
*   **Application Deadlines:** 
    *   Domestic Master's intake mentions specific dates like April 27, 2026, for post-admission documentation.
    *   International-specific deadlines are **not mentioned** in the provided text.
*   **Language Requirements:** Not mentioned in the provided text.
*   **Required Documents:** 
    *   Personnel files/Archives (人事档案) and "Real-life Performance" materials (本人现实表现) are mentioned for admitted students.
    *   General application documents for international students are **not mentioned**.
*   **Application Portal/Method:** 
    *   Postgraduate Admissions Website: [https://yzb.hitsz.edu.cn/zs](https://yzb.hitsz.edu.cn/zs)
    *   General English Website: [http://en.hitsz.edu.cn](http://en.hitsz.edu.cn)

## 4. Tuition, Fees, & Scholarships
*   **Application Fee:** Not mentioned in the provided text.
*   **Tuition Fee Breakdown:** Not mentioned in the provided text.
*   **Accommodation Fees:** Not mentioned in the provided text.
*   **Available Scholarships:**
    *   **Joint PhD Programs:** Mentions joint PhD recruitment with Great Bay University (大湾区大学) and Peng Cheng National Laboratory (鹏城国家实验室).
    *   **General Scholarships (CSC/Provincial):** Not mentioned in the provided text.

## 5. Contact Information & Next Steps
*   **International Students Office Contact:** 
    *   Phone: 0755-26033876 (General), 0755-86132651 (Huang Laoshi - Graduate Admissions).
    *   Address: HIT Campus of University Town of Shenzhen, Shenzhen, China, PC: 518055.
*   **Important Links:**
    *   Graduate Admissions Home: [https://yzb.hitsz.edu.cn/zs](https://yzb.hitsz.edu.cn/zs)
    *   English Home: [http://en.hitsz.edu.cn](http://en.hitsz.edu.cn)

## 6. Extraction Notes
*   The primary admissions portal (`zsb.hitsz.edu.cn`) was unreachable during the scrape (Connection Reset), which is likely why specific tuition and international scholarship details are missing.
*   The provided data focuses heavily on domestic postgraduate recruitment and faculty news.
*   **Missing Information:** International application deadlines, English-taught program confirmation, tuition amounts, and specific CSC scholarship procedures are absent from this specific data set.

---

## 7. Personalized Recommendations
*   **Matching Programs:** You should focus on the **School of Computer Science and Technology** and the **College of Artificial Intelligence**. These directly align with your interest in Computer Science and Data Analytics.
*   **Language Requirements:** You have an excellent IELTS score (8.0), which makes you highly competitive for English-taught programs. However, since you have **HSK 0**, you must verify if the CS Master's program is offered in English. If it is only in Chinese, you would typically need to complete 1 year of Chinese language training first.
*   **Scholarship Eligibility:** While the text mentions joint programs with National Labs, it does not detail the **Chinese Government Scholarship (CSC)**. However, as HITSZ is a prestigious campus, they typically host CSC candidates. You should contact **Huang Laoshi at 0755-86132651** to ask specifically about the "CSC Type B" application for Indonesian students.
*   **Budget/Cost:** The provided text does not list tuition. Note that top-tier STEM Master's in China often range from 30,000 to 45,000 RMB/year. Your budget of 40,000 RMB is likely sufficient for tuition, but you will need a full scholarship (like the CSC) to cover living expenses and accommodation in an expensive city like Shenzhen.